# 🤖 Análisis Exploratorio del Dataset: Twitter Profiles
**Objetivo:** Explorar el dataset crudo, identificar insconcistencias (valores nulos), definir un flujo de limpieza y la validacion comparando el dataset antes y despues.

**Variables claves**
* `name`: Nombre de la cuenta
* `followers_count`: Cantidad de seguidores de la cuenta
* `friends_count`: Cantidad de seguidos de la cuenta
* `post_count`: Cantidad de post hechos
* `lang`: Idioma de la cuenta
* `location`: Localizacion de la cuenta
* `default_profile_image`: Foto de perfil por defecto
* `profile_use_background_image`: Uso de foto de portada
* `verified`: Si la cuenta contiene el "verificado"
* `description`: Descripcion de la cuenta
* `created_at` : Fecha de la creacion de la cuenta
* `Y` **Variable objetivo** ¿La cuenta es una cuenta falsa (bot)? ('yes' or 'no').

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargamos el dataset
df = pd.read_csv('../data/raw/twitter_profiles_original.csv')

#Se muestra la informacion del dataset
print(f"Dimensiones del dataset: {df.shape}")
df.head()

Dimensiones del dataset: (7698, 13)


,name,screen_name,followers_count,friends_count,post_count,lang,location,default_profile_image,profile_use_background_image,verified,description,created_at,label
0,早川援,0918Bask,286.0,441.0,3431.0,ja,NaN,False,False,False,motorcycle.&car.,11-06-2013 11:20,0
1,ローラー,1120Roll,351.0,554.0,6179.0,ja,大正義横浜市,False,False,False,ゴロゴロするのが大好きなちなDeです。政治思想は掻い摘み派です。メモ帳化中。無言フォロー失礼...,13-05-2014 10:37,0
2,Kelsey Brown,keke_brooke,241.0,197.0,1874.0,en,NaN,False,True,False,Future mall Santa,04-05-2011 23:30,0
3,bio,pinkfunkys,2164.0,947.0,263152.0,en,NaN,False,True,False,no longer here I'm @toujoursbeiles,17-09-2010 14:02,0
4,Ms Kathy,191a5bd05da04dc,23.0,96.0,140.0,en,Wichita KS,False,True,False,Cosmetologist,06-02-2015 04:10,0


# El encuentro de valores nulos

Dentro de los dataset se encuentran varios valores nulos. Vamos a ver cuantos  existen para luego poder trabajar con ellos.

In [ ]:
df.isna().sum()

,0
name,0
screen_name,0
followers_count,3
friends_count,3
post_count,3
lang,3
location,1358
default_profile_image,3
profile_use_background_image,3
verified,3


En este caso la columna de 'location' se encuentran varios valores nulos escritos de la forma `NaN` y vamos a trabajar con ellos pero con el resto de columnas que tienen menos nulos vamos a hacer lo siguiente:
* `verified` es una variable que no deberia tener nulos ya que deberia especificar si tiene o no verificados = reemplazar por no verificado o eliminar outlier.
* `created_at` no tiene sentido que no haya fecha de creación = eliminar outliers.
* `followers/friends/post_count` cambiar NaN por 0
* `description` reemplazar los NaN por un texto donde se refiera a que el usuario no entrego una descripción de perfil

Finalmente, para la columna `location` transformar los datos a valores sin mayusculas, comas, y puntos, con tal de que el codigo entienda que son el mismo dato.

In [2]:
#Con esto nos encargamos de 'verified'
df['verified'] = df['verified'].fillna(False)

#Eliminamos las cuentas sin fechas
df = df.dropna(subset=['created_at'])

#Se eliminan los NaN por 0
cols_conteo = ['followers_count', 'friends_count', 'post_count']
df[cols_conteo] = df[cols_conteo].fillna(0)

#Se agrega un texto predeterminado a los Nan
df['description'] = df['description'].fillna('Usuario no entregó descripción de perfil')

#Verificamos que se hayan eliminado los nulos correspondientes
df.isna().sum()

/tmp/ipykernel_7204/3069675901.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['verified'] = df['verified'].fillna(False)


,0
name,0
screen_name,0
followers_count,0
friends_count,0
post_count,0
lang,0
location,1353
default_profile_image,0
profile_use_background_image,0
verified,0


Se aplica un filtro para eliminar ubicaciones únicas o extremadamente raras. Al convertir estos valores de baja frecuencia en nulos (NaN), limpiamos el "ruido" del dataset, permitiendo que el análisis se enfoque en zonas geográficas con una masa crítica de usuarios y facilitando la detección de patrones reales.

In [3]:
location_counts = df['location'].value_counts()
locations_appearing_once = location_counts[location_counts == 2] #Si la ubicacion no la comparten al menos 3 personas, no sera relevante

locations_to_remove = locations_appearing_once.index
df['location'] = df['location'].replace(locations_to_remove, np.nan)

print("Valores nulos después de eliminar locaciones de una sola aparición:")
print(df['location'].isna().sum())

Valores nulos después de eliminar locaciones de una sola aparición:
1765


Se cuantifica la cantidad de ubicaciones únicas para medir el nivel de "ruido" en el dataset antes de aplicar la limpieza final.

In [4]:
location_counts = df['location'].value_counts()
locations_appearing_once = location_counts[location_counts == 1]

print(f"El número de locaciones que aparecen solo una vez es: {len(locations_appearing_once)}")

El número de locaciones que aparecen solo una vez es: 2980


Se unifican variaciones de escritura de una misma ubicación (como "NYC" o "Australia, AU") bajo un nombre único. Asi agrupamos los datos correctamente para mejorar la precisión de los análisis y gráficos finales.

In [ ]:
# Importas tu herramienta (asegúrate de que la carpeta src sea tratada como módulo)
from src.procesamiento_texto import consolidar_ubicaciones

# Unificas New York
df = consolidar_ubicaciones(df, 'location', 'new york', 'new york')

# Unificas Australia
df = consolidar_ubicaciones(df, 'location', 'australia', 'australia')

# Unificas Italia
df = consolidar_ubicaciones(df, 'location', 'italy', 'italy')

# Verificas el resultado
display(df['location'].value_counts().head(10))

,count
location,
new york,297
London,67
Italy,64
"Sydney, Australia",51
California,48
Roma,47
Milano,43
Australia,41
Texas,40


Se generan variables binarias para medir la completitud del perfil, como la presencia de ubicación (has_location). Esto permite mostrar qué tan confiable es la información proporcionada por cada usuario y sera clave para la definicion de la variable `Y`

In [ ]:
from src.ingenieria_variables import generar_indicadores_perfil

df = generar_indicadores_perfil(df)

,count
has_location,
1,5916
0,1773


Se hace un proceso que elimina símbolos y caracteres especiales, dejando el texto listo para ser validado por la API geográfica sin errores de lectura.

In [ ]:
from src.validacion_geografica import limpiar_texto_regex

df['location_clean'] = df['location'].apply(limpiar_texto_regex)

Guardamos la nueva base de datos y procedemos a la verificacion y sus respectiva conclusion.

In [21]:
# Al final del Notebook de Limpieza
df.to_csv('../data/processed/twitter_profiles_cleaned.csv', index=False)